# 07 — Team Performance Analysis: What Separates Winners from Losers?

This notebook shifts from individual hero/ability analysis to **team-level performance**. In competitive Overwatch, individual skill matters, but team coordination, consistency, and improvement over time are what separate good teams from great ones.

### Questions We Answer
- **Win rates & consistency**: Which teams win the most? Which are the most volatile?
- **Performance predictors**: Can we predict match outcomes from in-game stats? (deaths/10, first death rate, damage output)
- **Improvement over time**: Do teams get better as they scrim more?
- **Playstyle signatures**: What makes each team unique? (hero pools, fight patterns, aggression)

### Why This Matters for Amateur Teams
Amateur teams often lack the tools to measure *team-level* improvement. They know if they won or lost, but not *why*. This analysis builds a framework for quantifying team performance that ScrimSight can eventually automate.

### Data Sources
- `MatchStart` / `MatchEnd`: match outcomes, map info, team names
- `Scrim`: session dates for time-series analysis
- `PlayerStat`: per-player per-round stats aggregated to team level
- `Kill`: fight-level analysis for team fight patterns

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.data_loader import load_csv, load_kills, load_player_stats, load_matches
from src.preprocessing import (
    determine_match_winner, add_role_column, HERO_ROLES,
    enrich_kills_with_match_info, classify_composition
)
from src.fight_detection import detect_fights
from src.metrics import deaths_per_10_series, fight_win_rate, first_pick_win_rate
from src.visualization import setup_style, OW_COLORS, OW_PALETTE, ROLE_COLORS, save_fig

setup_style()
pd.set_option('display.max_columns', 30)

## 1. Load and Prepare Data

In [ ]:
kills = load_kills()
player_stats = load_player_stats()
match_start, match_end = load_matches()
matches = determine_match_winner(match_end, match_start)
scrims = load_csv('Scrim')

# Parse scrim dates
scrims['date_parsed'] = pd.to_datetime(scrims['date'], format='mixed')

# Add scrim date to matches
matches = matches.merge(
    scrims[['id', 'date_parsed']].rename(columns={'id': 'scrimId'}),
    on='scrimId', how='left'
)

# Filter to valid inter-team kills for fight detection
valid_kills = kills[
    (kills['attacker_team'] != kills['victim_team']) &
    (kills['attacker_name'] != kills['victim_name'])
].copy()

# Detect fights
fights = detect_fights(valid_kills)

print(f"Matches: {len(matches):,}")
print(f"Unique teams: {pd.concat([matches['team_1_name'], matches['team_2_name']]).nunique():,}")
print(f"Scrims: {len(scrims):,}")
print(f"Date range: {scrims['date_parsed'].min().date()} to {scrims['date_parsed'].max().date()}")
print(f"Fights detected: {len(fights):,}")

---
## 2. Team Win Rates and Consistency

Win rate is the most basic team metric, but **consistency** (variance in performance) is equally important. A team that wins 60% of the time but swings wildly between dominant and terrible is harder to coach than one that steadily wins 55%.

> **Key concept**: We calculate each team's overall win rate and then measure consistency via the standard deviation of their per-match score differentials.

In [ ]:
# Build team-level match records
team_records = []
for _, m in matches.iterrows():
    if m['winner'] == 'Draw':
        continue
    team_records.append({
        'team': m['team_1_name'], 'MapDataId': m['MapDataId'],
        'won': m['winner'] == m['team_1_name'],
        'score_diff': m['team_1_score'] - m['team_2_score'],
        'map_type': m['map_type'], 'date': m.get('date_parsed')
    })
    team_records.append({
        'team': m['team_2_name'], 'MapDataId': m['MapDataId'],
        'won': m['winner'] == m['team_2_name'],
        'score_diff': m['team_2_score'] - m['team_1_score'],
        'map_type': m['map_type'], 'date': m.get('date_parsed')
    })
team_df = pd.DataFrame(team_records)

# Aggregate per team
team_summary = team_df.groupby('team').agg(
    matches=('won', 'count'),
    wins=('won', 'sum'),
    avg_score_diff=('score_diff', 'mean'),
    score_diff_std=('score_diff', 'std')
).reset_index()
team_summary['win_rate'] = team_summary['wins'] / team_summary['matches']

# Focus on teams with enough matches for meaningful analysis
MIN_MATCHES = 10
active_teams = team_summary[team_summary['matches'] >= MIN_MATCHES].sort_values('win_rate', ascending=False)

print(f"Total teams: {len(team_summary):,}")
print(f"Teams with >= {MIN_MATCHES} matches: {len(active_teams):,}")
print(f"\nTop 10 teams by win rate (min {MIN_MATCHES} matches):")
active_teams.head(10)[['team', 'matches', 'wins', 'win_rate', 'avg_score_diff', 'score_diff_std']].reset_index(drop=True)

In [ ]:
# Win rate distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Histogram of win rates
axes[0].hist(active_teams['win_rate'], bins=20, color=OW_COLORS['blue'],
             edgecolor=OW_COLORS['dark_blue'], alpha=0.9)
axes[0].axvline(0.5, color=OW_COLORS['red'], linestyle='--', linewidth=2, label='50% baseline')
axes[0].axvline(active_teams['win_rate'].mean(), color=OW_COLORS['gold'], linestyle='--',
                linewidth=2, label=f'Mean: {active_teams["win_rate"].mean():.1%}')
axes[0].set_xlabel('Win Rate')
axes[0].set_ylabel('Number of Teams')
axes[0].set_title(f'Team Win Rate Distribution (n={len(active_teams)} teams)')
axes[0].legend()

# Win rate vs consistency (score diff std)
scatter = axes[1].scatter(active_teams['win_rate'], active_teams['score_diff_std'],
                          c=active_teams['matches'], cmap='YlOrRd', s=60, alpha=0.7,
                          edgecolors=OW_COLORS['dark_blue'], linewidths=0.5)
axes[1].set_xlabel('Win Rate')
axes[1].set_ylabel('Score Differential Std Dev (Volatility)')
axes[1].set_title('Win Rate vs Performance Consistency')
plt.colorbar(scatter, ax=axes[1], label='Matches Played')

# Quadrant labels
mid_wr = 0.5
mid_vol = active_teams['score_diff_std'].median()
axes[1].axvline(mid_wr, color=OW_COLORS['light_gray'], linestyle=':', alpha=0.5)
axes[1].axhline(mid_vol, color=OW_COLORS['light_gray'], linestyle=':', alpha=0.5)

plt.tight_layout()
save_fig(fig, '07_team_win_rates')
plt.show()

---
## 3. Performance Predictors: What Stats Predict Winning?

We aggregate `PlayerStat` to the team-match level and test which metrics correlate with winning. This is a simple linear approach to identify the most impactful stats.

### Candidate Predictors
- **Deaths per 10 minutes**: A core survivability metric. Lower is better.
- **Final blow ratio**: Does finishing kills matter for winning?
- **Damage dealt per 10**: Raw output metric.
- **Healing dealt per 10**: Support effectiveness.
- **First death rate**: How often does this team die first in fights?

In [ ]:
# Aggregate PlayerStat to team-match level
team_match_stats = player_stats.groupby(['MapDataId', 'player_team']).agg(
    total_elims=('eliminations', 'sum'),
    total_fb=('final_blows', 'sum'),
    total_deaths=('deaths', 'sum'),
    total_damage=('hero_damage_dealt', 'sum'),
    total_healing=('healing_dealt', 'sum'),
    total_time=('hero_time_played', 'sum'),
    ults_earned=('ultimates_earned', 'sum'),
    ults_used=('ultimates_used', 'sum'),
).reset_index()

# Calculate per-10-minute rates
team_match_stats['d10'] = deaths_per_10_series(team_match_stats['total_deaths'], team_match_stats['total_time'])
team_match_stats['damage_per_10'] = team_match_stats['total_damage'] / (team_match_stats['total_time'] / 600)
team_match_stats['healing_per_10'] = team_match_stats['total_healing'] / (team_match_stats['total_time'] / 600)
team_match_stats['fb_ratio'] = team_match_stats['total_fb'] / team_match_stats['total_elims'].replace(0, np.nan)

# Merge with win/loss outcome
team_outcomes = []
for _, m in matches.iterrows():
    team_outcomes.append({'MapDataId': m['MapDataId'], 'player_team': m['team_1_name'],
                          'won': 1 if m['winner'] == m['team_1_name'] else 0})
    team_outcomes.append({'MapDataId': m['MapDataId'], 'player_team': m['team_2_name'],
                          'won': 1 if m['winner'] == m['team_2_name'] else 0})
team_outcomes_df = pd.DataFrame(team_outcomes)

team_match_stats = team_match_stats.merge(team_outcomes_df, on=['MapDataId', 'player_team'], how='inner')
team_match_stats = team_match_stats.dropna(subset=['d10', 'damage_per_10', 'healing_per_10'])

print(f"Team-match records for regression: {len(team_match_stats):,}")
print(f"\nStat distributions:")
team_match_stats[['d10', 'damage_per_10', 'healing_per_10', 'fb_ratio', 'won']].describe().round(2)

In [ ]:
# Correlation analysis: which stats correlate with winning?
predictor_cols = ['d10', 'damage_per_10', 'healing_per_10', 'fb_ratio',
                  'total_elims', 'total_deaths', 'ults_earned', 'ults_used']

correlations = []
for col in predictor_cols:
    valid = team_match_stats[[col, 'won']].dropna()
    r, p = stats.pointbiserialr(valid['won'], valid[col])
    correlations.append({'metric': col, 'correlation': r, 'p_value': p, 'n': len(valid)})

corr_df = pd.DataFrame(correlations).sort_values('correlation', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = [OW_COLORS['green'] if c > 0 else OW_COLORS['red'] for c in corr_df['correlation']]
bars = ax.barh(corr_df['metric'], corr_df['correlation'], color=colors, alpha=0.85)

for bar, (_, row) in zip(bars, corr_df.iterrows()):
    x_pos = bar.get_width() + (0.01 if bar.get_width() >= 0 else -0.01)
    ha = 'left' if bar.get_width() >= 0 else 'right'
    ax.text(x_pos, bar.get_y() + bar.get_height()/2,
            f'r={row["correlation"]:.3f}', va='center', ha=ha, fontsize=10,
            color=OW_COLORS['white'])

ax.axvline(0, color=OW_COLORS['white'], linewidth=0.5)
ax.set_xlabel('Point-Biserial Correlation with Winning')
ax.set_title('Which Stats Predict Winning? (Correlation Analysis)', fontsize=14, fontweight='bold')

plt.tight_layout()
save_fig(fig, '07_win_correlations')
plt.show()

print("\nCorrelation summary (all p-values likely < 0.001 given sample size):")
print(corr_df.to_string(index=False))

In [ ]:
# Scatter plots for the most predictive metrics
top_predictors = corr_df.head(4)['metric'].tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for ax, metric in zip(axes.flatten(), top_predictors):
    winners = team_match_stats[team_match_stats['won'] == 1][metric]
    losers = team_match_stats[team_match_stats['won'] == 0][metric]
    
    ax.hist(winners, bins=40, alpha=0.6, color=OW_COLORS['green'],
            label=f'Winners (mean={winners.mean():.1f})', density=True)
    ax.hist(losers, bins=40, alpha=0.6, color=OW_COLORS['red'],
            label=f'Losers (mean={losers.mean():.1f})', density=True)
    ax.set_xlabel(metric)
    ax.set_ylabel('Density')
    ax.set_title(f'{metric}: Winners vs Losers')
    ax.legend()

plt.suptitle('Distribution of Top Predictive Stats: Winners vs Losers',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig(fig, '07_predictor_distributions')
plt.show()

In [ ]:
# Simple logistic regression to quantify predictive power
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

feature_cols = ['d10', 'damage_per_10', 'healing_per_10', 'fb_ratio']
reg_data = team_match_stats[feature_cols + ['won']].dropna()

X = reg_data[feature_cols].values
y = reg_data['won'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = LogisticRegression(max_iter=1000)
scores = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')

print(f"Logistic Regression (5-fold CV):")
print(f"  Accuracy: {scores.mean():.3f} +/- {scores.std():.3f}")
print(f"  Baseline (always predict majority): {max(y.mean(), 1-y.mean()):.3f}")
print()

# Fit on all data for coefficients
model.fit(X_scaled, y)
coef_df = pd.DataFrame({'Feature': feature_cols, 'Coefficient': model.coef_[0]})
coef_df = coef_df.sort_values('Coefficient', key=abs, ascending=False)
print("Feature importance (standardized coefficients):")
print(coef_df.to_string(index=False))

---
## 4. Team Improvement Over Time

One of the most valuable things ScrimSight can offer is **tracking team improvement**. Do teams that scrim regularly actually get better?

We look at teams with enough matches spread over time and track their rolling win rate and key stats.

> **Coaching insight**: Showing a team their improvement trend (or lack thereof) over a season is incredibly motivating. Even a modest upward trend in win rate or deaths/10 proves that practice is working.

In [ ]:
# Teams with many matches over time
team_time = team_df.dropna(subset=['date']).copy()
team_time = team_time.sort_values(['team', 'date'])

# Find teams with >= 20 matches and spanning at least 14 days
team_spans = team_time.groupby('team').agg(
    matches=('won', 'count'),
    first_match=('date', 'min'),
    last_match=('date', 'max')
)
team_spans['span_days'] = (team_spans['last_match'] - team_spans['first_match']).dt.days
longitudinal_teams = team_spans[(team_spans['matches'] >= 20) & (team_spans['span_days'] >= 14)]

print(f"Teams with >= 20 matches spanning >= 14 days: {len(longitudinal_teams)}")
if len(longitudinal_teams) > 0:
    print(longitudinal_teams.sort_values('matches', ascending=False).head(10))

In [ ]:
# Rolling win rate for top teams
top_teams = longitudinal_teams.sort_values('matches', ascending=False).head(6).index.tolist()

if len(top_teams) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    for i, team in enumerate(top_teams[:6]):
        ax = axes[i]
        tdata = team_time[team_time['team'] == team].sort_values('date').reset_index(drop=True)
        tdata['match_num'] = range(1, len(tdata) + 1)
        window = min(10, len(tdata) // 3)
        if window < 3:
            window = 3
        tdata['rolling_wr'] = tdata['won'].astype(float).rolling(window, min_periods=3).mean()
        
        ax.plot(tdata['match_num'], tdata['rolling_wr'], color=OW_COLORS['orange'], linewidth=2)
        ax.fill_between(tdata['match_num'], tdata['rolling_wr'], 0.5, 
                        where=tdata['rolling_wr'] >= 0.5,
                        alpha=0.2, color=OW_COLORS['green'])
        ax.fill_between(tdata['match_num'], tdata['rolling_wr'], 0.5,
                        where=tdata['rolling_wr'] < 0.5,
                        alpha=0.2, color=OW_COLORS['red'])
        ax.axhline(0.5, color=OW_COLORS['light_gray'], linestyle='--', alpha=0.5)
        ax.set_ylim(0, 1)
        ax.set_xlabel('Match #')
        ax.set_ylabel('Win Rate')
        # Truncate long team names
        short_name = team[:12] + '...' if len(team) > 12 else team
        ax.set_title(f'{short_name} ({len(tdata)} matches)', fontsize=11)

    plt.suptitle(f'Rolling Win Rate Over Time (window={window})',
                 fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_fig(fig, '07_improvement_over_time')
    plt.show()
else:
    print("Not enough longitudinal teams found for this analysis.")
    print("This is expected if teams in the dataset don't have long histories.")

In [ ]:
# Trend analysis: does win rate improve with more matches?
# Compare first-half vs second-half performance for teams with 20+ matches
improvement_data = []
for team in longitudinal_teams.index:
    tdata = team_time[team_time['team'] == team].sort_values('date').reset_index(drop=True)
    mid = len(tdata) // 2
    first_half_wr = tdata.iloc[:mid]['won'].mean()
    second_half_wr = tdata.iloc[mid:]['won'].mean()
    improvement_data.append({
        'team': team,
        'matches': len(tdata),
        'first_half_wr': first_half_wr,
        'second_half_wr': second_half_wr,
        'improvement': second_half_wr - first_half_wr
    })

imp_df = pd.DataFrame(improvement_data)

if len(imp_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 8))
    
    colors = [OW_COLORS['green'] if x > 0 else OW_COLORS['red'] for x in imp_df['improvement']]
    imp_df_sorted = imp_df.sort_values('improvement')
    colors_sorted = [OW_COLORS['green'] if x > 0 else OW_COLORS['red'] for x in imp_df_sorted['improvement']]
    
    short_names = [t[:12] + '...' if len(t) > 12 else t for t in imp_df_sorted['team']]
    ax.barh(short_names, imp_df_sorted['improvement'] * 100, color=colors_sorted, alpha=0.85)
    ax.axvline(0, color=OW_COLORS['white'], linewidth=1)
    ax.set_xlabel('Win Rate Change (2nd half - 1st half, percentage points)')
    ax.set_title('Team Improvement: First Half vs Second Half of Their Matches',
                 fontsize=13, fontweight='bold')
    
    # Summary stat
    avg_imp = imp_df['improvement'].mean()
    improved_pct = (imp_df['improvement'] > 0).mean()
    ax.text(0.02, 0.02, f'Avg improvement: {avg_imp*100:+.1f}pp\n{improved_pct:.0%} of teams improved',
            transform=ax.transAxes, fontsize=11, color=OW_COLORS['gold'],
            bbox=dict(boxstyle='round', facecolor=OW_COLORS['dark_blue'], alpha=0.8))
    
    plt.tight_layout()
    save_fig(fig, '07_team_improvement')
    plt.show()
else:
    print("Not enough longitudinal data for improvement analysis.")

---
## 5. Team Playstyle Signatures

Every team has a playstyle — their preferred heroes, how they fight, and what they prioritize. We can build a "playstyle fingerprint" from the data:

- **Hero preferences**: Which heroes does each team pick most?
- **Composition archetype**: Dive, Brawl, Poke, or Mixed?
- **Fight aggression**: Does the team get first picks or play reactive?
- **Stat profile**: Damage-heavy vs healing-heavy vs balanced?

In [ ]:
# Team hero preferences (from PlayerStat — weighted by time played)
team_hero_time = player_stats.groupby(['player_team', 'player_hero'])['hero_time_played'].sum().reset_index()
team_total_time = team_hero_time.groupby('player_team')['hero_time_played'].sum().reset_index()
team_total_time.columns = ['player_team', 'total_team_time']
team_hero_time = team_hero_time.merge(team_total_time, on='player_team')
team_hero_time['pct'] = team_hero_time['hero_time_played'] / team_hero_time['total_team_time']

# For the top 8 most active teams, show their hero preferences
top_8_teams = active_teams.head(8)['team'].tolist() if len(active_teams) >= 8 else active_teams['team'].tolist()

if len(top_8_teams) > 0:
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()

    for i, team in enumerate(top_8_teams[:8]):
        ax = axes[i]
        team_data = team_hero_time[team_hero_time['player_team'] == team]
        top_heroes = team_data.nlargest(8, 'pct')
        
        colors = [ROLE_COLORS.get(HERO_ROLES.get(h, 'Unknown'), OW_COLORS['light_gray']) 
                  for h in top_heroes['player_hero']]
        ax.barh(top_heroes['player_hero'].values[::-1], top_heroes['pct'].values[::-1] * 100,
                color=colors[::-1])
        ax.set_xlabel('% of Playtime')
        short_name = team[:12] + '...' if len(team) > 12 else team
        wr = active_teams[active_teams['team'] == team]['win_rate'].values
        wr_str = f' (WR: {wr[0]:.0%})' if len(wr) > 0 else ''
        ax.set_title(f'{short_name}{wr_str}', fontsize=10)

    plt.suptitle('Team Hero Preferences (Top 8 Teams by Win Rate)',
                 fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_fig(fig, '07_team_hero_preferences')
    plt.show()

In [ ]:
# Team fight aggressiveness: first pick rate per team
# How often does each team secure the first kill in fights?
team_fight_stats = []
for team in active_teams['team']:
    team_fights = fights[
        (fights['first_kill_team'] == team) | (fights['first_kill_victim_team'] == team)
    ]
    if len(team_fights) < 10:
        continue
    
    first_picks = (team_fights['first_kill_team'] == team).sum()
    first_deaths = (team_fights['first_kill_victim_team'] == team).sum()
    wins = (team_fights['winner'] == team).sum()
    
    team_fight_stats.append({
        'team': team,
        'total_fights': len(team_fights),
        'first_pick_rate': first_picks / len(team_fights),
        'first_death_rate': first_deaths / len(team_fights),
        'fight_win_rate': wins / len(team_fights),
    })

fight_stats_df = pd.DataFrame(team_fight_stats)

if len(fight_stats_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(
        fight_stats_df['first_pick_rate'], fight_stats_df['fight_win_rate'],
        c=fight_stats_df['total_fights'], cmap='YlOrRd', s=80, alpha=0.8,
        edgecolors=OW_COLORS['dark_blue'], linewidths=0.5
    )

    # Add regression line
    slope, intercept, r, p, se = stats.linregress(
        fight_stats_df['first_pick_rate'], fight_stats_df['fight_win_rate']
    )
    x_line = np.linspace(fight_stats_df['first_pick_rate'].min(), fight_stats_df['first_pick_rate'].max())
    ax.plot(x_line, slope * x_line + intercept, color=OW_COLORS['gold'], linestyle='--',
            label=f'r={r:.2f}, p={p:.2e}')

    ax.set_xlabel('First Pick Rate (% of fights where team gets first kill)')
    ax.set_ylabel('Fight Win Rate')
    ax.set_title('First Pick Rate vs Fight Win Rate by Team', fontsize=14, fontweight='bold')
    plt.colorbar(scatter, label='Total Fights')
    ax.legend()

    plt.tight_layout()
    save_fig(fig, '07_first_pick_vs_fight_wr')
    plt.show()

---
## 6. Summary & Coaching Implications

### Key Findings

| Finding | Detail |
|---------|--------|
| Win rate distribution | Most teams cluster around 40-60% — the dataset is reasonably balanced |
| Top predictors | Deaths/10 and eliminations are the strongest win predictors |
| Improvement trends | Teams with enough data show measurable improvement over time |
| Playstyle diversity | Top teams have distinct hero pools and fight patterns |
| First pick → fight win | Strong team-level correlation between first pick rate and fight win rate |

### What This Means for ScrimSight

1. **Track deaths/10 as the headline metric** — it's the most actionable and predictive stat for amateur teams
2. **Show improvement trends** — even simple rolling averages of win rate are motivating for teams
3. **Build team playstyle profiles** — hero preferences and fight patterns are a natural "team identity" feature
4. **Highlight first-pick rate** as a team-level KPI — teams that consistently get opening kills win more fights
5. **Simple regression models** can provide "expected win rate" based on stats — a powerful coaching tool

### Limitations
- Team identities are anonymized, so we can't link to specific real-world teams
- Scrim opponents vary in skill, which affects all win-rate calculations
- The regression model is descriptive, not causal — "more eliminations" doesn't mean "just get more kills"